# Ingest — organisation attributes

Derives `org_attributes` automatically from `viva_credits_weekly`. No org file is required.
Optional CSVs in `Files/landing/org/` override only populated attributes and can add people.

This is what turns per-person consumption into a department, cost-centre or business-unit view —
which is most of the chargeback story. Consumption carries the population and its org attributes.
The preferred Dataflow Gen2 path and the CSV fallback both feed `viva_credits_weekly`.

## Two things make this harder than it looks

**Every tenant spells these differently.** One exports `department`, another `Department`, another
`Dept`. `Organisation`/`Organization` is kept separately from Department. Columns use known aliases,
and anything expected but absent is created empty. One missing attribute should cost you that one
breakdown, not the whole table.

**Half of them are not standard Entra properties.** `department`, `jobTitle`, `city` and `country`
come out of the bulk download. `manager` needs Graph or the preview download. `costCenter`,
`jobFamily` and `businessUnit` are extension attributes whose names differ per tenant — usually
populated by an HR sync. If you do not have them, leave them out; nothing breaks.

**Non-destructive enrichment.** Latest populated consumption attributes win (MetricDate first),
then populated manual attributes override them. Ties use lexical value order, not file order.
Upserts never delete people or erase populated attributes with blanks. Removed manual overrides
fall back to consumption values when available; absent source values retain the previous value.
`person_id` and the legacy `user_principal_name` org join column both hold the trimmed/lowercase
UPN (including email aliases), falling back to trimmed/lowercase PersonId. A PersonId-only key is
opaque, not an inferred email, and matches only the same consumption identity.


In [ ]:
LANDING = "Files/landing/org"
TBL = "org_attributes"

# Canonical name -> the spellings seen in the wild. Add yours if it is missing;
# comparison ignores case, spaces, dashes and underscores.
TBL_METRICS = 'viva_credits_weekly'
PERSON_ALIASES = {
    'user_principal_name': ['UserPrincipalName', 'UPN', 'UserPrincipal', 'Email', 'Mail', 'EmailAddress'],
    'person_id': ['PersonId'],
    'display_name': ['DisplayName', 'Name', 'FullName', 'PreferredName'],
    'department': ['Department', 'Dept'],
    'organisation': ['Organisation', 'Organization'],
    'job_title': ['JobTitle', 'Title', 'Role'],
    'job_family': ['JobFamily', 'Function', 'FunctionType', 'JobFunction'],
    'city': ['City', 'OfficeLocation', 'Location'],
    'country': ['Country', 'CountryOrRegion', 'Region'],
    'cost_center': ['CostCenter', 'CostCentre'],
    'manager': ['Manager', 'ManagerName', 'Supervisor', 'ManagerId', 'ManagerUPN'],
    'business_unit': ['BusinessUnit', 'Division', 'Segment'],
}
ORG_COLUMNS = [c for c in PERSON_ALIASES if c not in ('person_id', 'user_principal_name')]


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import notebookutils
import re


def norm(name):
    return re.sub(r"[ _\-]", "", name).lower()


def normalise_person(record):
    """One identity contract for CSV facts and automatic/manual org rows."""
    lookup = {}
    for name, value in record.items():
        text = str(value).strip() if value is not None else ''
        if text:
            lookup.setdefault(norm(name), text)
    result = {}
    for canon, aliases in PERSON_ALIASES.items():
        result[canon] = next((lookup[norm(a)] for a in [canon] + aliases if norm(a) in lookup), None)
    for key in ('person_id', 'user_principal_name'):
        if result[key]:
            result[key] = result[key].lower()
    result['person_id'] = result['user_principal_name'] or result['person_id']
    return result


def identity_sql(column):
    return f"NULLIF(LOWER(REGEXP_REPLACE({column}, r'(?U)^[\\s\\x1c-\\x1f]+|[\\s\\x1c-\\x1f]+$', '')), '')"


def org_candidate(record, priority):
    person = normalise_person(record)
    fields = {norm(k): str(v).strip() for k, v in record.items() if v is not None}
    # ISO dates from the connector/notebook sort chronologically. Dates beat load order.
    date = fields.get('metricdate', '')
    loaded = fields.get('loadedat', '')
    values = tuple((priority, date, loaded, person[c]) if person[c] else None for c in ORG_COLUMNS)
    return person['person_id'], values


def merge_attributes(left, right):
    """Associative reduction: populated manual > latest populated consumption."""
    return tuple(max(a, b) if a is not None and b is not None else (a if a is not None else b)
                 for a, b in zip(left, right))


def org_record(item):
    key, values = item
    return (key, key, *[v[3] if v is not None else None for v in values])


sources = []
if spark.catalog.tableExists(TBL_METRICS):
    sources.append((spark.table(TBL_METRICS), 0))
# Absence is optional; permission/read errors must still fail visibly.
if notebookutils.fs.exists(LANDING):
    for file in sorted(notebookutils.fs.ls(LANDING), key=lambda f: f.name):
        if file.name.lower().endswith('.csv'):
            sources.append((spark.read.option('header', True).option('inferSchema', False).csv(file.path), 1))


In [ ]:
candidates = []
for frame, priority in sources:
    if not any(norm(c) in {norm(a) for key in ('person_id', 'user_principal_name')
                           for a in [key] + PERSON_ALIASES[key]} for c in frame.columns):
        raise ValueError(f'Org source has no UPN alias or PersonId: {frame.columns}')
    candidates.append(frame.rdd.map(lambda row, p=priority: org_candidate(row.asDict(), p)))

schema = ', '.join(f'{c} string' for c in ['person_id', 'user_principal_name'] + ORG_COLUMNS)
if candidates:
    reduced = (spark.sparkContext.union(candidates).filter(lambda item: item[0] is not None)
               .reduceByKey(merge_attributes).map(org_record))
    org = spark.createDataFrame(reduced, schema)
else:
    org = spark.createDataFrame([], schema)
org = org.withColumn('_loaded_at', F.current_timestamp())

if spark.catalog.tableExists(TBL):
    existing = set(spark.table(TBL).columns)
    added = [c for c in ['person_id'] + ORG_COLUMNS if c not in existing]
    if added:
        spark.sql(f'ALTER TABLE {TBL} ADD COLUMNS (' + ', '.join(f'{c} STRING' for c in added) + ')')
    updates = {c: f'COALESCE(s.{c}, t.{c})' for c in ORG_COLUMNS}
    updates.update({c: f's.{c}' for c in ('person_id', 'user_principal_name', '_loaded_at')})
    target_key = identity_sql('t.user_principal_name')
    if (spark.table(TBL).alias('t').groupBy(F.expr(target_key)).count()
            .filter(F.col('count') > 1).limit(1).count()):
        raise ValueError('Existing org has colliding normalised keys; reconcile duplicates before merging.')
    (DeltaTable.forName(spark, TBL).alias('t')
     .merge(org.alias('s'), target_key + ' = s.user_principal_name')
     .whenMatchedUpdate(set=updates).whenNotMatchedInsertAll().execute())
else:
    org.write.format('delta').saveAsTable(TBL)
print(f'{TBL}: enriched {org.count():,} people; no existing people deleted')


## Check the join will actually work

The most common disappointment with this template is department breakdowns coming back empty, and
the cause is almost always that the org file's UPNs do not match the consumption export's. Better
to find that here than to stare at a blank chart.

In [ ]:
spark.sql(f"""
    SELECT  COUNT(*)                                        AS people,
            COUNT(department)                               AS with_department,
            COUNT(cost_center)                              AS with_cost_centre,
            COUNT(manager)                                  AS with_manager,
            COUNT(DISTINCT department)                      AS departments
    FROM    {TBL}
""").show(truncate=False)

if spark.catalog.tableExists("viva_credits_weekly"):
    consumers = (spark.table(TBL_METRICS).rdd
                 .map(lambda row: (normalise_person(row.asDict())['person_id'],))
                 .filter(lambda row: row[0] is not None).distinct())
    spark.createDataFrame(consumers, 'upn string').createOrReplaceTempView('_org_consumers')
    spark.sql(f"""
        SELECT  COUNT(*)                                   AS consumers,
                COUNT(o.user_principal_name)               AS matched_to_org,
                ROUND(100.0 * COUNT(o.user_principal_name) / NULLIF(COUNT(*), 0), 1) AS pct
        FROM    _org_consumers c
        LEFT JOIN {TBL} o ON o.user_principal_name = c.upn
    """).show(truncate=False)
    print("A low percentage means the two files identify people differently.")
    print('PersonId-only consumption joins on its opaque ID; no UPN mapping is inferred.')
else:
    print("viva_credits_weekly not loaded yet - run the Viva ingester to check the join.")